In [83]:
import math
import torch
import collections
from itertools import combinations
import numpy as np

In [11]:
a = [0, 1, 2, 3, 4]

In [ ]:
import itertools
import torch

In [84]:
from itertools import combinations, chain
from functools import cache

Tensor = torch.Tensor

@cache
def count_ways(n: int) -> int:
    """Memoized counting (useful if called repeatedly)."""
    return max(0, (1 << n - 1) - 1) if n >= 2 else 0

def partitions(n: int):
    """Generator with walrus operator for clarity."""
    if n < 2:
        return
    items = set(range(1, n))
    yield from (
        [list((0, *c)), list(items - set(c))]
        for r in range(n)
        if (rest := n - 1 - r) > 0
        for c in combinations(items, r)
    )

def _get_unfold_transpose_order(d: int, mode: int) -> list[int]:
    enum = list(range(d))
    order = [mode] + enum[:mode] + enum[mode + 1 :]

    return order


def unfold(X: Tensor, mode: int) -> Tensor:
    # n1 x ... x nd -> n_mode x (N/n_mode), N = n1 * ... * nd
    order = _get_unfold_transpose_order(X.ndim, mode)

    x = X.permute(*order)
    x = x.reshape((x.shape[0], -1))

    return x


def fold(x: Tensor, original_shape: tuple[int], mode: int):
    order = _get_unfold_transpose_order(len(original_shape), mode)
    inv_order = np.argsort(order).tolist()
    perm_shape = tuple(original_shape[j] for j in order)

    # a = A.reshape(np.array(original_shape)[order])
    # a = np.transpose(a, inv_order)

    X = x.reshape(*perm_shape)
    X = X.permute(*inv_order)

    return X

def invert_permutation(perm: list[int]) -> list[int]:
    """Return the inverse of a permutation."""
    inv = [0] * len(perm)
    for i, p in enumerate(perm):
        inv[p] = i
    return inv

In [109]:
c = torch.randn(10, 3, 4, 5, 6)

In [110]:
_get_unfold_transpose_order(c.ndim, 3)

[3, 0, 1, 2, 4]

In [111]:
torch.argsort(torch.tensor(_get_unfold_transpose_order(c.ndim, 3)))

tensor([1, 2, 3, 0, 4])

In [112]:
reshapes = [*partitions(5)]

In [117]:
ranks = [2, 3, 2, 4, 2]

In [118]:
from operator import itemgetter

In [ ]:
from typing import Any


itemgetter[tuple[Any, Any, *tuple[Any, ...]]](*reversed(np.argsort(c.shape).tolist()))(ranks)

(2, 2, 4, 2, 3)

In [116]:
ranks

[2, 3, 2, 2, 2]

In [99]:
for reshape in reshapes:
    print(reshape)
    perm = [*chain.from_iterable(reshape)]
    tmp = c.permute(*perm)
    perm_shape = tmp.shape
    print(tmp.shape)
    prod_shape = math.prod([c.shape[i] for i in reshape[0]])
    tmp = tmp.reshape(prod_shape, -1)
    print(tmp.shape)
    inv_order = invert_permutation([*chain.from_iterable(reshape)])
    print(inv_order)
    tmp = tmp.reshape(*perm_shape)
    tmp = tmp.permute(*inv_order)
    print(tmp.shape)
    print("Done", end="\n\n")
    

[[0], [1, 2, 3, 4]]
torch.Size([10, 3, 4, 5, 6])
torch.Size([10, 360])
[0, 1, 2, 3, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 1], [2, 3, 4]]
torch.Size([10, 3, 4, 5, 6])
torch.Size([30, 120])
[0, 1, 2, 3, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 2], [1, 3, 4]]
torch.Size([10, 4, 3, 5, 6])
torch.Size([40, 90])
[0, 2, 1, 3, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 3], [1, 2, 4]]
torch.Size([10, 5, 3, 4, 6])
torch.Size([50, 72])
[0, 2, 3, 1, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 4], [1, 2, 3]]
torch.Size([10, 6, 3, 4, 5])
torch.Size([60, 60])
[0, 2, 3, 4, 1]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 1, 2], [3, 4]]
torch.Size([10, 3, 4, 5, 6])
torch.Size([120, 30])
[0, 1, 2, 3, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 1, 3], [2, 4]]
torch.Size([10, 3, 5, 4, 6])
torch.Size([150, 24])
[0, 1, 3, 2, 4]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 1, 4], [2, 3]]
torch.Size([10, 3, 6, 4, 5])
torch.Size([180, 20])
[0, 1, 3, 4, 2]
torch.Size([10, 3, 4, 5, 6])
Done

[[0, 2, 3], [1, 4]]
torch.S